# 00 — Famílias de equações estacionárias por construção

## Resumo

Este estudo identifica famílias de equações capazes de gerar séries temporais estacionárias por construção e avalia quais delas são mais adequadas ao benchmark. A análise separa a **restrição sobre o mecanismo** da **inicialização estacionária**: uma recorrência estável iniciada em um valor arbitrário pode conter um transiente, enquanto uma trajetória estritamente estacionária exige que o estado inicial já siga a distribuição invariante ou que seja construída a partir de um passado infinito. São consideradas médias móveis finitas, AR/ARMA, VAR e sistemas lineares, recorrências não lineares contrativas, processos Gaussianos com kernels estacionários e GARCH. Pequenas verificações numéricas confirmam as condições matemáticas; elas servem como testes de consistência, não como substitutos das garantias analíticas. A conclusão recomenda começar por AR/VAR parametrizados dentro da região estacionária e inicializados pela covariância de Lyapunov, seguido por recorrências não lineares contrativas. Essas famílias preservam lags e coeficientes explícitos, facilitando a produção de ground truths estruturais, assinados, locais e dependentes do horizonte.


## 1. Pergunta e critério de seleção

**Pergunta.** Quais mecanismos discretos permitem sortear parâmetros dentro de um domínio que garante estacionaridade e, ao mesmo tempo, conservam relações temporais explícitas para gerar ground-truth explanations?

Uma família é especialmente útil ao projeto quando satisfaz quatro propriedades: (1) a condição de estacionaridade pode ser verificada antes da simulação; (2) a inicialização estacionária é conhecida ou aproximável de forma controlada; (3) variáveis e lags permanecem identificáveis; e (4) o mecanismo permite derivar explicações para cada horizonte.


## 2. O que significa garantir estacionaridade

Uma série é **estritamente estacionária** quando qualquer distribuição conjunta permanece igual após deslocar todos os índices no tempo. Ela é **fracamente estacionária** quando a média é constante e a covariância depende somente da distância entre dois instantes. Para processos Gaussianos, essas duas definições coincidem ([Heaps, 2023, Sec. 1, pp. 74–75](https://doi.org/10.1080/10618600.2022.2079648)).

> **Condição do mecanismo + condição inicial = trajetória estacionária desde o primeiro ponto observado.**

A condição do mecanismo impede que o efeito do passado cresça indefinidamente. A condição inicial remove o transiente. Quando a distribuição estacionária puder ser calculada, o estado inicial será sorteado diretamente dela. Quando isso não for possível, um *burn-in* pode aproximá-la, mas essa aproximação deverá ser registrada como tal. Testes como ADF ou KPSS serão apenas diagnósticos de amostras finitas; eles não constituem a garantia do gerador.

Também é importante separar **estacionariedade** de **não explosão**. Uma trajetória limitada pode ter média variável no tempo, e uma distribuição estacionária de cauda pesada pode ocasionalmente produzir valores grandes.


## 3. Notação comum

| Símbolo | Significado |
|---|---|
| $X_t$ | valor univariado no instante $t$ |
| $\mathbf{x}_t$ | vetor com todas as séries no instante $t$ |
| $\varepsilon_t$ ou $\boldsymbol{\varepsilon}_t$ | inovação i.i.d., independente do passado |
| $\ell$ | lag, com $\ell\geq 1$ |
| $B$ | operador de atraso, $BX_t=X_{t-1}$ |
| $\rho(A)$ | raio espectral: maior módulo dos autovalores de $A$ |
| $\lVert\cdot\rVert$ | norma especificada para vetor ou matriz |
| $k(\tau)$ | covariância estacionária em função apenas do deslocamento $\tau$ |


In [1]:
from __future__ import annotations

import importlib.metadata
import json
import platform

import numpy as np
import pandas as pd
from scipy.linalg import solve_discrete_lyapunov
from statsmodels.tsa.stattools import levinson_durbin_pacf

ROOT_SEED = 20261006
rng = np.random.default_rng(ROOT_SEED)
configuration = {
    'root_seed': ROOT_SEED,
    'purpose': 'literature-backed design study before generator implementation',
    'families': ['MA', 'AR/ARMA', 'VAR/state space', 'nonlinear contraction', 'stationary GP', 'GARCH'],
    'python': platform.python_version(),
    'numpy': importlib.metadata.version('numpy'),
    'pandas': importlib.metadata.version('pandas'),
    'scipy': importlib.metadata.version('scipy'),
    'statsmodels': importlib.metadata.version('statsmodels'),
}
print(json.dumps(configuration, indent=2, ensure_ascii=False))

{
  "root_seed": 20261006,
  "purpose": "literature-backed design study before generator implementation",
  "families": [
    "MA",
    "AR/ARMA",
    "VAR/state space",
    "nonlinear contraction",
    "stationary GP",
    "GARCH"
  ],
  "python": "3.11.9",
  "numpy": "1.26.4",
  "pandas": "2.1.4",
  "scipy": "1.11.4",
  "statsmodels": "0.15.0"
}


**O que foi feito.** A célula fixa uma semente local somente para as verificações numéricas e registra as versões utilizadas. Nenhum estado aleatório global do programa do usuário é alterado.


## 4. Famílias lineares univariadas

### 4.1 Média móvel finita — MA($q$)

$$X_t=\mu+\sum_{j=0}^{q}\theta_j\varepsilon_{t-j}.$$

Com inovações i.i.d. definidas em todos os tempos, qualquer vetor deslocado de observações é a mesma função de um vetor de inovações com a mesma distribuição. Portanto, o processo é estritamente estacionário sem restrição adicional sobre os $\theta_j$. É uma opção simples e com memória finita, mas não representa recursão do valor observado.

### 4.2 AR(1)

$$X_t-\mu=\phi(X_{t-1}-\mu)+\varepsilon_t,\qquad |\phi|<1.$$

Para inovação Gaussiana de variância $\sigma_\varepsilon^2$, a inicialização exata é

$$X_0\sim\mathcal N\!\left(\mu,\frac{\sigma_\varepsilon^2}{1-\phi^2}\right).$$

### 4.3 AR($p$) e ARMA($p,q$)

$$\Phi(B)(X_t-\mu)=\Theta(B)\varepsilon_t,\qquad \Phi(z)=1-\sum_{j=1}^{p}\phi_jz^j.$$

A parte autorregressiva é causal e estacionária quando todas as raízes de $\Phi(z)=0$ estão **fora** do círculo unitário. De modo equivalente, os autovalores da matriz companheira ficam **dentro** do círculo unitário. A diferença entre “fora” e “dentro” vem apenas da convenção de polinômio utilizada. As raízes de $\Theta$ controlam a invertibilidade da representação MA, enquanto a estacionariedade é controlada por $\Phi$. Shao e Wu apresentam a forma equivalente em autovalores e a representação MA infinita com decaimento geométrico ([2007, Teorema 5.2, p. 1783](https://doi.org/10.1214/009053606000001479)).

Em vez de sortear $\phi_j$ e rejeitar combinações instáveis, pode-se sortear coeficientes de autocorrelação parcial $\kappa_j\in(-1,1)$ e convertê-los em coeficientes AR. Monahan propõe essa reparametrização para restringir implicitamente modelos ARMA à região estacionária/invertível ([1984, pp. 403–404](https://doi.org/10.1093/biomet/71.2.403)).


In [2]:
def roots_of_ar_lag_polynomial(phi: np.ndarray) -> np.ndarray:
    # Phi(z) = 1 - phi_1 z - ... - phi_p z^p
    return np.polynomial.polynomial.polyroots(np.r_[1.0, -np.asarray(phi, dtype=float)])

def ar_is_stationary(phi: np.ndarray, tolerance: float = 1e-12) -> bool:
    return bool(np.all(np.abs(roots_of_ar_lag_polynomial(phi)) > 1.0 + tolerance))

partial_autocorrelations = np.array([0.60, -0.40, 0.30])
pacf_result = levinson_durbin_pacf(np.r_[1.0, partial_autocorrelations])
phi_from_pacf = np.asarray(pacf_result.arcoefs)

ar_candidates = {
    'AR(1) válido': np.array([0.70]),
    'AR(2) válido': np.array([0.60, -0.20]),
    'AR(1) explosivo': np.array([1.10]),
    'AR(3) via PACF': phi_from_pacf,
}
ar_rows = []
for name, phi in ar_candidates.items():
    roots = roots_of_ar_lag_polynomial(phi)
    ar_rows.append({
        'candidate': name,
        'phi': np.round(phi, 4).tolist(),
        'min_root_modulus': float(np.min(np.abs(roots))),
        'stationary': ar_is_stationary(phi),
    })
ar_checks = pd.DataFrame(ar_rows)
assert ar_checks.loc[ar_checks['candidate'] == 'AR(1) válido', 'stationary'].item()
assert not ar_checks.loc[ar_checks['candidate'] == 'AR(1) explosivo', 'stationary'].item()
assert ar_checks.loc[ar_checks['candidate'] == 'AR(3) via PACF', 'stationary'].item()
display(ar_checks)
print('PACF sorteada:', partial_autocorrelations)
print('Coeficientes AR obtidos:', np.round(phi_from_pacf, 4))

,candidate,phi,min_root_modulus,stationary
0,AR(1) válido,[0.7],1.428571,True
1,AR(2) válido,"[0.6, -0.2]",2.236068,True
2,AR(1) explosivo,[1.1],0.909091,False
3,AR(3) via PACF,"[0.96, -0.652, 0.3]",1.483824,True


PACF sorteada: [ 0.6 -0.4  0.3]
Coeficientes AR obtidos: [ 0.96  -0.652  0.3  ]


**O que foi feito.** As raízes do polinômio de lag distinguem exemplos admissíveis e inadmissíveis. A última linha demonstra a estratégia preferida para AR($p$): sortear valores PACF dentro de $(-1,1)$ e transformá-los, evitando uma busca aleatória por coeficientes estáveis.


## 5. VAR e sistema linear multivariado

Para VAR(1),

$$\mathbf{x}_{t+1}=A\mathbf{x}_t+\boldsymbol{\varepsilon}_{t+1},\qquad \rho(A)<1.$$

Se $\boldsymbol{\varepsilon}_t\sim\mathcal N(0,D)$, a covariância estacionária $C$ é a solução única da equação discreta de Lyapunov

$$C=ACA^\top+D.$$

A condição $\rho(A)<1$ e a equação de Lyapunov são apresentadas por Oku e Aihara ([2018, Sec. 2.1, pp. 167–168](https://doi.org/10.1587/nolta.9.166)). Sortear $\mathbf{x}_0\sim\mathcal N(0,C)$ produz estacionariedade desde o primeiro ponto. Para VAR($p$), empilham-se os $p$ estados em uma matriz companheira e aplica-se a mesma condição.

Para amostragem multivariada de ordem maior, Heaps descreve uma correspondência entre VAR estacionários e matrizes de autocorrelação parcial, seguida de uma transformação para matrizes sem restrições ([2023, Sec. 2, pp. 75–77](https://doi.org/10.1080/10618600.2022.2079648)). Essa alternativa é mais sofisticada que simplesmente normalizar uma matriz VAR(1), mas evita rejeição e preserva estruturas interpretáveis.


In [3]:
raw_A = rng.normal(size=(2, 2))
raw_radius = float(np.max(np.abs(np.linalg.eigvals(raw_A))))
target_radius = 0.80
A = raw_A * (target_radius / raw_radius)
D = np.array([[0.30, 0.08], [0.08, 0.20]])
C = solve_discrete_lyapunov(A, D)
lyapunov_residual = float(np.linalg.norm(C - A @ C @ A.T - D, ord='fro'))
minimum_covariance_eigenvalue = float(np.min(np.linalg.eigvalsh(C)))
initial_state = rng.multivariate_normal(mean=np.zeros(2), cov=C)

var_check = pd.Series({
    'spectral_radius': float(np.max(np.abs(np.linalg.eigvals(A)))),
    'lyapunov_residual': lyapunov_residual,
    'minimum_covariance_eigenvalue': minimum_covariance_eigenvalue,
    'sampled_stationary_initial_state_x1': float(initial_state[0]),
    'sampled_stationary_initial_state_x2': float(initial_state[1]),
})
assert var_check['spectral_radius'] < 1.0
assert lyapunov_residual < 1e-10
assert minimum_covariance_eigenvalue > 0.0
display(var_check.to_frame('value'))

,value
spectral_radius,8.000000e-01
lyapunov_residual,2.810038e-16
minimum_covariance_eigenvalue,5.643038e-01
sampled_stationary_initial_state_x1,1.247742e+00
sampled_stationary_initial_state_x2,8.185765e-02


**O que foi feito.** Uma matriz bivariada foi escalada para raio espectral $0{,}8$. A solução de Lyapunov é positiva definida e seu resíduo numérico é praticamente zero; ela fornece a distribuição do estado inicial para uma trajetória Gaussiana exatamente estacionária.


## 6. Recorrências não lineares contrativas

Considere

$$X_{t+1}=R(X_t,\ldots,X_{t-p+1};\varepsilon_{t+1}).$$

Shao e Wu mostram que existe uma solução estacionária quando $R$ possui limites de Lipschitz coordenados $a_j\geq0$ cuja soma é menor que um, sob uma condição de momento da inovação ([2007, Teorema 5.1, pp. 1782–1783](https://doi.org/10.1214/009053606000001479)):

$$|R(\mathbf{x};\varepsilon)-R(\mathbf{x}';\varepsilon)|\leq\sum_{j=1}^{p}a_j|x_j-x'_j|,\qquad \sum_{j=1}^{p}a_j<1.$$

Uma família diretamente amostrável é

$$X_t=b+\sum_{j=1}^{p}w_j\tanh(X_{t-\ell_j})+\varepsilon_t,\qquad \sum_j|w_j|<1,$$

porque $\tanh$ é 1-Lipschitz. Ela mantém lags esparsos, sinais e não linearidade explícitos. Produtos arbitrários como $X_{t-1}X_{t-2}$ não satisfazem uma cota global de Lipschitz em $\mathbb R^2$; portanto, não podem entrar nessa família sem limites adicionais sobre o estado ou uma transformação contrativa comprovada.


In [4]:
lags = np.array([1, 3, 6])
weights = np.array([0.35, -0.20, 0.15])
contraction_budget = float(np.sum(np.abs(weights)))
nonlinear_candidate = pd.Series({
    'formula': '0.2 + 0.35*tanh(LAG(1)) - 0.20*tanh(LAG(3)) + 0.15*tanh(LAG(6)) + epsilon',
    'lags': lags.tolist(),
    'coordinate_lipschitz_bounds': np.abs(weights).tolist(),
    'sum_of_bounds': contraction_budget,
    'analytic_condition_holds': contraction_budget < 1.0,
})
assert contraction_budget < 1.0
display(nonlinear_candidate.to_frame('value'))

,value
formula,0.2 + 0.35*tanh(LAG(1)) - 0.20*tanh(LAG(3)) + ...
lags,"[1, 3, 6]"
coordinate_lipschitz_bounds,"[0.35, 0.2, 0.15]"
sum_of_bounds,0.7
analytic_condition_holds,True


**O que foi feito.** A soma dos limites de sensibilidade aos três lags é $0{,}70$, abaixo de um. A verificação é feita sobre a própria fórmula e pode futuramente virar um certificado anexado à AST.


## 7. Processos Gaussianos com covariância estacionária

$$X(t)\sim\mathcal{GP}(\mu,k(t-t')).$$

Com média constante e um kernel positivo semidefinido que depende somente de $\tau=t-t'$, o processo Gaussiano é estacionário. Pelo teorema de Bochner, um kernel estacionário é a transformada de Fourier de uma medida espectral positiva. Wilson e Adams usam esse resultado para construir kernels de mistura espectral ([2013, Sec. 3, pp. 1069–1070](https://proceedings.mlr.press/v28/wilson13.pdf)):

$$k_{SM}(\tau)=\sum_{q=1}^{Q}w_q\exp(-2\pi^2v_q\tau^2)\cos(2\pi\mu_q\tau),\qquad w_q,v_q>0.$$

Também podem ser usados RBF, Matérn e periodicidade estacionária. Essa família é eficiente para diversidade espectral e se aproxima do tipo de prior utilizado por geradores baseados em kernels. Sua explicação natural descreve componentes de covariância/frequência; ela não fornece automaticamente uma máscara esparsa variável–lag como uma equação AR.


In [5]:
time = np.arange(64, dtype=float)
tau = time[:, None] - time[None, :]
rbf = np.exp(-0.5 * (tau / 6.0) ** 2)
periodic = np.exp(-2.0 * np.sin(np.pi * tau / 12.0) ** 2 / 0.8**2)
spectral_mixture = (
    0.65 * np.exp(-2.0 * np.pi**2 * 0.002 * tau**2) * np.cos(2.0 * np.pi * tau / 12.0)
    + 0.35 * np.exp(-2.0 * np.pi**2 * 0.010 * tau**2) * np.cos(2.0 * np.pi * tau / 4.0)
)
kernel_checks = pd.DataFrame([
    {'kernel': name, 'minimum_eigenvalue': float(np.min(np.linalg.eigvalsh(matrix)))}
    for name, matrix in [('RBF', rbf), ('periodic', periodic), ('spectral mixture', spectral_mixture)]
])
assert kernel_checks['minimum_eigenvalue'].min() > -1e-10
display(kernel_checks)

,kernel,minimum_eigenvalue
0,RBF,-2.022531e-15
1,periodic,-7.926682e-15
2,spectral mixture,6.230554e-02


**O que foi feito.** As matrizes de covariância geradas por três kernels estacionários são positivas semidefinidas, admitindo apenas pequenos autovalores negativos de arredondamento. Isso verifica a construção no conjunto finito de 64 instantes; a garantia geral vem da validade analítica dos kernels.


## 8. GARCH para volatilidade estacionária

$$X_t=\sigma_tZ_t,\qquad \sigma_t^2=\omega+\alpha X_{t-1}^2+\beta\sigma_{t-1}^2,$$

com $\omega>0$, $\alpha,\beta\geq0$ e $Z_t$ i.i.d. padronizado. A condição necessária e suficiente para solução estritamente estacionária é

$$\mathbb E[\log(\alpha Z_t^2+\beta)]<0.$$

Para $\mathbb E[Z_t^2]=1$, a restrição simples $\alpha+\beta<1$ é suficiente e também fornece segundo momento finito. A condição logarítmica e sua extensão para GARCH são discutidas por Francq e Zakoïan ([2012, versão de trabalho, Secs. 1 e 5, pp. 3 e 14](https://mpra.ub.uni-muenchen.de/22414/1/MPRA_paper_22414.pdf)). Essa família é útil quando a explicação desejada envolve o mecanismo da **variância condicional**. Ela deve ser avaliada separadamente das explicações da média prevista.


In [6]:
omega, alpha, beta = 0.10, 0.08, 0.88
z = rng.standard_normal(200_000)
estimated_log_condition = float(np.mean(np.log(alpha * z**2 + beta)))
garch_check = pd.Series({
    'omega': omega,
    'alpha': alpha,
    'beta': beta,
    'alpha_plus_beta': alpha + beta,
    'finite_variance_sufficient_condition': alpha + beta < 1.0,
    'monte_carlo_E_log(alpha_Z2_plus_beta)': estimated_log_condition,
})
assert omega > 0.0 and alpha >= 0.0 and beta >= 0.0
assert alpha + beta < 1.0
assert estimated_log_condition < 0.0
display(garch_check.to_frame('value'))

,value
omega,0.1
alpha,0.08
beta,0.88
alpha_plus_beta,0.96
finite_variance_sufficient_condition,True
monte_carlo_E_log(alpha_Z2_plus_beta),-0.046621


**O que foi feito.** O exemplo satisfaz a restrição analítica simples $\alpha+\beta=0{,}96<1$. A esperança logarítmica estimada é mostrada apenas como conferência numérica; a garantia utilizada para aceitar os parâmetros é a desigualdade analítica.


In [7]:
candidates = pd.DataFrame([
    {
        'family': 'MA(q)',
        'pre_simulation_guarantee': 'finite function of iid innovations',
        'exact_initialization': 'innovations also sampled before t=0',
        'multivariate': 'yes',
        'nonlinear': 'no',
        'natural_explanation_gt': 'innovation-lag coefficients/contributions',
        'priority': 2,
    },
    {
        'family': 'AR/ARMA',
        'pre_simulation_guarantee': 'AR roots outside unit circle; PACF parameterization',
        'exact_initialization': 'stationary state covariance for Gaussian innovations',
        'multivariate': 'via VARMA',
        'nonlinear': 'no',
        'natural_explanation_gt': 'variable-lag mask, signed coefficients, local and horizon effects',
        'priority': 1,
    },
    {
        'family': 'VAR/state space',
        'pre_simulation_guarantee': 'companion spectral radius below one',
        'exact_initialization': 'discrete Lyapunov covariance',
        'multivariate': 'native',
        'nonlinear': 'no',
        'natural_explanation_gt': 'source-lag graph, matrices and powers across horizons',
        'priority': 1,
    },
    {
        'family': 'contractive nonlinear AR',
        'pre_simulation_guarantee': 'sum of coordinate Lipschitz bounds below one',
        'exact_initialization': 'coupling from the past or controlled burn-in',
        'multivariate': 'possible, requires a matrix-norm certificate',
        'nonlinear': 'yes',
        'natural_explanation_gt': 'structure, Jacobians, local contributions and propagated gradients',
        'priority': 2,
    },
    {
        'family': 'stationary Gaussian process',
        'pre_simulation_guarantee': 'constant mean and valid lag-only covariance',
        'exact_initialization': 'joint Gaussian sample has no recurrence boundary',
        'multivariate': 'yes, with valid cross-covariance',
        'nonlinear': 'not an explicit recurrence',
        'natural_explanation_gt': 'kernel and spectral-component attribution',
        'priority': 3,
    },
    {
        'family': 'GARCH(1,1)',
        'pre_simulation_guarantee': 'E log(alpha Z^2 + beta) below zero; alpha+beta below one is sufficient',
        'exact_initialization': 'sample invariant volatility or use controlled burn-in',
        'multivariate': 'extensions exist but are more complex',
        'nonlinear': 'variance recursion',
        'natural_explanation_gt': 'conditional-variance mechanism',
        'priority': 4,
    },
])
display(candidates.sort_values(['priority', 'family']).reset_index(drop=True))

,family,pre_simulation_guarantee,exact_initialization,multivariate,nonlinear,natural_explanation_gt,priority
0,AR/ARMA,AR roots outside unit circle; PACF parameteriz...,stationary state covariance for Gaussian innov...,via VARMA,no,"variable-lag mask, signed coefficients, local ...",1
1,VAR/state space,companion spectral radius below one,discrete Lyapunov covariance,native,no,"source-lag graph, matrices and powers across h...",1
2,MA(q),finite function of iid innovations,innovations also sampled before t=0,yes,no,innovation-lag coefficients/contributions,2
3,contractive nonlinear AR,sum of coordinate Lipschitz bounds below one,coupling from the past or controlled burn-in,"possible, requires a matrix-norm certificate",yes,"structure, Jacobians, local contributions and ...",2
4,stationary Gaussian process,constant mean and valid lag-only covariance,joint Gaussian sample has no recurrence boundary,"yes, with valid cross-covariance",not an explicit recurrence,kernel and spectral-component attribution,3
5,"GARCH(1,1)",E log(alpha Z^2 + beta) below zero; alpha+beta...,sample invariant volatility or use controlled ...,extensions exist but are more complex,variance recursion,conditional-variance mechanism,4


**O que foi feito.** A tabela transforma as fórmulas pesquisadas em alternativas de projeto. A prioridade considera simultaneamente garantia matemática, facilidade de inicialização, compatibilidade multivariada e qualidade do ground truth explicativo.


## 9. Decisão para a implementação posterior

A implementação deve começar com duas rotas complementares:

1. **AR($p$)/VAR($p$) estacionário por parametrização.** Para o caso univariado, sortear PACFs em um intervalo com margem, por exemplo $[-0{,}95,0{,}95]$, e convertê-las em coeficientes AR. Para VAR, começar com VAR(1) de raio espectral controlado e avançar para a parametrização por matrizes de autocorrelação parcial. Calcular a distribuição inicial pela equação de Lyapunov.
2. **AR não linear contrativo.** Restringir inicialmente as não linearidades a funções com constante de Lipschitz conhecida, como $\tanh$, e normalizar o orçamento $\sum_j|w_j|$ para ficar abaixo de $1-\delta$, onde $\delta>0$ é uma margem configurável.

MA/ARMA pode ser acrescentado em seguida. Kernels estacionários formam uma rota paralela de geração espectral. GARCH deve entrar quando o benchmark definir explicitamente a explicação da variância condicional.

### Consequência para a gramática atual

Uma gramática estacionária precisa carregar um **certificado da família inteira**, e não validar cada nó isoladamente. O terminal determinístico `TIME(t)` e tendências lineares mudam a média com $t$ e ficam fora desta gramática. `ADD` e `MULTIPLY` continuam disponíveis apenas quando a composição preserva a condição da família: somas ponderadas de termos contrativos podem ser certificadas; produtos arbitrários entre lags não. O mecanismo poderá registrar `family`, `stationarity_condition`, `margin`, `initialization_method` e os valores usados no certificado.

### Relação com os ground truths

Para AR/VAR, a estrutura variável–lag e os coeficientes assinados vêm diretamente da equação; os efeitos de múltiplos horizontes são propagados pelas potências da matriz companheira. Para recorrências contrativas, os Jacobianos locais propagam sensibilidades através da recursão. Em ambos os casos, ainda será necessário distinguir contribuição do mecanismo verdadeiro, comportamento do modelo treinado e explicação produzida pelo método XAI.


## 10. Referências verificadas

- Monahan, J. F. (1984). *A Note on Enforcing Stationarity in Autoregressive-Moving Average Models*. Biometrika, 71(2), 403–404. DOI: [10.1093/biomet/71.2.403](https://doi.org/10.1093/biomet/71.2.403).
- Shao, X.; Wu, W. B. (2007). *Asymptotic Spectral Theory for Nonlinear Time Series*. The Annals of Statistics, 35(4), 1773–1801. DOI: [10.1214/009053606000001479](https://doi.org/10.1214/009053606000001479).
- Oku, M.; Aihara, K. (2018). *On the Covariance Matrix of the Stationary Distribution of a Noisy Dynamical System*. NOLTA, IEICE, 9(2), 166–184. DOI: [10.1587/nolta.9.166](https://doi.org/10.1587/nolta.9.166).
- Heaps, S. E. (2023). *Enforcing Stationarity through the Prior in Vector Autoregressions*. Journal of Computational and Graphical Statistics, 32(1), 74–83. DOI: [10.1080/10618600.2022.2079648](https://doi.org/10.1080/10618600.2022.2079648).
- Wilson, A. G.; Adams, R. P. (2013). *Gaussian Process Kernels for Pattern Discovery and Extrapolation*. ICML/PMLR 28, 1067–1075. [Paper](https://proceedings.mlr.press/v28/wilson13.html).
- Francq, C.; Zakoïan, J.-M. (2012). *Strict Stationarity Testing and Estimation of Explosive and Stationary Generalized Autoregressive Conditional Heteroscedasticity Models*. Econometrica, 80(2), 821–861. DOI: [10.3982/ECTA9407](https://doi.org/10.3982/ECTA9407).

As mesmas entradas foram adicionadas a `references/library.bib`, a bibliografia canônica do repositório.


## Registro completo do estudo

- **Pergunta de pesquisa:** quais famílias de equações discretas permitem estacionariedade por construção e ground truths explicativos úteis?
- **Hipótese:** AR/VAR parametrizados na região estacionária e recorrências contrativas oferecem o melhor equilíbrio inicial entre garantia, diversidade e interpretabilidade.
- **Configuração:** seis famílias analisadas; semente local `20261006`; exemplos AR(1), AR(2), AR(3)-PACF, VAR(1) bivariado, AR não linear com três lags, três kernels estacionários e GARCH(1,1).
- **Proveniência:** fontes primárias listadas na Seção 10; código executado no commit-base `cda3a52af76e517bf2e5d9168adc33ee38904c57`, com alterações de trabalho ainda não commitadas.
- **Uso da semente:** `numpy.random.default_rng(20261006)` é privado ao notebook e controla somente matriz VAR, estado inicial e diagnóstico Monte Carlo do GARCH. As verificações de raízes, contração e kernels são determinísticas.
- **Métricas/verificações:** módulo mínimo das raízes AR, raio espectral, resíduo de Lyapunov, menor autovalor da covariância, soma dos limites de Lipschitz, menor autovalor dos kernels e condição GARCH.
- **Resumo científico:** AR/VAR fornece a primeira rota de implementação; a inicialização deve fazer parte do certificado. Recorrências `tanh` com orçamento contrativo fornecem uma extensão não linear segura. GP estacionário e GARCH devem ser tratados como famílias com tipos próprios de ground truth. Ausência de explosão em uma simulação não é utilizada como prova de estacionariedade.
- **Artefatos:** somente este notebook executado; nenhum dataset ou sidecar foi criado.
- **Reprodução exata a partir da raiz:** `.\.venv\Scripts\python.exe -m jupyter nbconvert --to notebook --execute --inplace experiments\abstract_syntax_tree\stationarity_by_construction\00_stationary_equation_families.ipynb --ExecutePreprocessor.timeout=180`
